### Ανάλυση Παλινδρόμησης για το Red Wine Quality Dataset

Αυτό το notebook υλοποιεί ολόκληρο το 10-φασικό pipeline μηχανικής μάθησης για το σύνολο δεδομένων **Wine Quality Red** από το UCI.

- Διατηρούμε **και τις 1599 εγγραφές**
- **Δεν αφαιρούμε διπλότυπα**, επειδή θεωρούνται επιστημονικά έγκυρες επαναλαμβανόμενες παρατηρήσεις
- Χρησιμοποιούμε **random_state=42** όπου είναι δυνατό
- Ο στόχος `quality` αντιμετωπίζεται ως συνεχής μεταβλητή για πρόβλημα παλινδρόμησης


In [ ]:
!pip install numpy pandas scikit-learn matplotlib seaborn scipy shap -q
!pip install shap xgboost tabulate -q
import os
from pathlib import Path

# Δημιουργία κατάλογων
BASE_DIR = Path('/content/wine_quality_analysis')
for d in ['data', 'figures', 'results', 'scripts', 'report']:
    (BASE_DIR / d).mkdir(parents=True, exist_ok=True)

print("✅ Δημιουργήθηκαν όλοι οι κατάλογοι")

✅ Δημιουργήθηκαν όλοι οι κατάλογοι


In [ ]:
# Εισαγωγή βιβλιοθηκών
import os
import json
import random
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from scipy import stats
from sklearn.base import clone
from sklearn.feature_selection import RFE, SelectFromModel, mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, RandomizedSearchCV, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVR

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

BASE_DIR = Path('/home/ubuntu/wine_quality_analysis')
DATA_DIR = BASE_DIR / 'data'
FIG_DIR = BASE_DIR / 'figures'
RES_DIR = BASE_DIR / 'results'
UCI_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
DATA_PATH = DATA_DIR / 'winequality-red.csv'


### Φάση 1 — EDA
Εξετάζουμε τη δομή των δεδομένων, τα βασικά στατιστικά, τις συσχετίσεις και τα outliers.

In [ ]:
import os

# Φόρτωση δεδομένων από το UCI
DATA_DIR.mkdir(parents=True, exist_ok=True) # Ensure the data directory exists
urlretrieve(UCI_URL, DATA_PATH)
df = pd.read_csv(DATA_PATH, sep=';')
features = [c for c in df.columns if c != 'quality']
X = df[features].copy()
y = df['quality'].copy()

print(df.shape)
print(df.head())
print('Διπλότυπα:', df.duplicated().sum())
print(df['quality'].value_counts().sort_index())

# Περιγραφικά στατιστικά
summary = df.describe().T
summary['median'] = df.median(numeric_only=True)
summary['skew'] = df.skew(numeric_only=True)
summary['kurtosis'] = df.kurtosis(numeric_only=True)
summary

(1599, 12)
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  
0      9.4        5  
1      9.8        5  
2      9.

,count,mean,std,min,25%,50%,75%,max,median,skew,kurtosis
fixed acidity,1599.0,8.319637,1.741096,4.60000,7.1000,7.90000,9.200000,15.90000,7.90000,0.982751,1.132143
volatile acidity,1599.0,0.527821,0.179060,0.12000,0.3900,0.52000,0.640000,1.58000,0.52000,0.671593,1.225542
citric acid,1599.0,0.270976,0.194801,0.00000,0.0900,0.26000,0.420000,1.00000,0.26000,0.318337,-0.788998
residual sugar,1599.0,2.538806,1.409928,0.90000,1.9000,2.20000,2.600000,15.50000,2.20000,4.540655,28.617595
chlorides,1599.0,0.087467,0.047065,0.01200,0.0700,0.07900,0.090000,0.61100,0.07900,5.680347,41.715787
free sulfur dioxide,1599.0,15.874922,10.460157,1.00000,7.0000,14.00000,21.000000,72.00000,14.00000,1.250567,2.023562
total sulfur dioxide,1599.0,46.467792,32.895324,6.00000,22.0000,38.00000,62.000000,289.00000,38.00000,1.515531,3.809824
density,1599.0,0.996747,0.001887,0.99007,0.9956,0.99675,0.997835,1.00369,0.99675,0.071288,0.934079
pH,1599.0,3.311113,0.154386,2.74000,3.2100,3.31000,3.400000,4.01000,3.31000,0.193683,0.806943
sulphates,1599.0,0.658149,0.169507,0.33000,0.5500,0.62000,0.730000,2.00000,0.62000,2.428672,11.720251


### Φάση 2 — Προεπεξεργασία
Κάνουμε split 80/20 και χρησιμοποιούμε StandardScaler μέσα σε Pipeline ώστε να αποφύγουμε leakage.

In [ ]:
# Διαχωρισμός train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

# Ορισμός pipelines για όλα τα baseline μοντέλα
def get_models():
    return {
        'LinearRegression': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
        'RandomForest': Pipeline([('model', RandomForestRegressor(random_state=SEED))]),
        'GradientBoosting': Pipeline([('model', GradientBoostingRegressor(random_state=SEED))]),
        'SVR': Pipeline([('scaler', StandardScaler()), ('model', SVR())]),
        'ElasticNet': Pipeline([('scaler', StandardScaler()), ('model', ElasticNet(random_state=SEED, max_iter=10000))]),
        'KNN': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsRegressor())]),
    }

models = get_models()
X_train.shape, X_test.shape


((1279, 11), (320, 11))

### Φάσεις 3 έως 9
Εκπαίδευση baseline μοντέλων, 10-fold CV, feature selection, tuning, residual analysis, SHAP και τελική σύγκριση.

In [ ]:
# Για το πλήρες reproducible execution, εκτελούμε το αποθηκευμένο script
# ΠΡΟΣΟΧΗ ΘΕΛΕΙ ΧΕΙΡΟΚΙΝΗΤΟ UPLOAD ΣΤΑ FILES, ΕΚΕΙ ΠΟΥ ΘΑ ΛΕΕΙ SCRIPTS ΓΙΑ ΝΑ ΤΡΕΞΕΙ ΣΤΟ ΣΩΣΤΟ PATH
%run /content/wine_quality_analysis/scripts/full_pipeline.py


Base directory: /content/wine_quality_analysis
Running Phase 1 - EDA...
Saved: /content/wine_quality_analysis/results/descriptive_statistics.csv
Saved: /content/wine_quality_analysis/results/target_distribution.csv
Saved: /content/wine_quality_analysis/results/correlation_matrix.csv
Saved: /content/wine_quality_analysis/figures/phase1_correlation_heatmap.png
Saved: /content/wine_quality_analysis/figures/phase1_boxplots_features.png
Saved: /content/wine_quality_analysis/figures/phase1_target_distribution.png
Saved: /content/wine_quality_analysis/figures/phase1_pairplot.png
Saved: /content/wine_quality_analysis/results/iqr_outlier_summary.csv
Saved: /content/wine_quality_analysis/results/iqr_outlier_flags.csv
Saved: /content/wine_quality_analysis/figures/phase1_iqr_outlier_counts.png
Done with EDA!
Running Phase 2 - Preprocessing...
Saved: /content/wine_quality_analysis/report/preprocessing_documentation.md
Saved: /content/wine_quality_analysis/results/split_summary.csv
Running Phase 3 -

### Φάση 10 — Παραδοτέα
Τα αποτελέσματα έχουν αποθηκευτεί στους φακέλους `figures/`, `results/` και `report/`.

In [ ]:
# Δείτε τα παραγόμενα αρχεία
import os
for folder in ['figures', 'results', 'report']:
    print(f"\n📁 {folder}/")
    for file in os.listdir(f'/content/wine_quality_analysis/{folder}'):
        print(f"  ✓ {file}")

# Κατέβασμα όλων
import shutil
from google.colab import files
shutil.make_archive('/content/wine_quality_results', 'zip', '/content/wine_quality_analysis')
files.download('/content/wine_quality_results.zip')



📁 figures/
  ✓ phase8_shap_dependence_alcohol.png
  ✓ phase1_target_distribution.png
  ✓ phase9_model_radar_chart.png
  ✓ phase8_shap_dependence_sulphates.png
  ✓ phase1_boxplots_features.png
  ✓ phase9_baseline_rmse_comparison.png
  ✓ phase9_permutation_importance_final_model.png
  ✓ phase7_residuals_vs_predicted.png
  ✓ phase8_shap_summary.png
  ✓ phase8_shap_bar.png
  ✓ phase7_residual_distribution.png
  ✓ phase8_shap_dependence_volatile_acidity.png
  ✓ phase7_qq_plot.png
  ✓ phase1_correlation_heatmap.png
  ✓ phase1_iqr_outlier_counts.png
  ✓ phase1_pairplot.png
  ✓ phase7_actual_vs_predicted.png
  ✓ .ipynb_checkpoints

📁 results/
  ✓ iqr_outlier_summary.csv
  ✓ descriptive_statistics.csv
  ✓ final_model_metrics.csv
  ✓ tuned_metrics.csv
  ✓ mutual_information_ranking.csv
  ✓ shap_importance.csv
  ✓ pearson_feature_ranking.csv
  ✓ feature_selection_results.csv
  ✓ final_comparison.csv
  ✓ baseline_metrics.csv
  ✓ residuals.csv
  ✓ permutation_importance_final_model.csv
  ✓ feature

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>